In [57]:
# import motoman_config
import sys
sys.path.insert(0,'/usr/local/lib/python3.11/site-packages')

import pinocchio as pin
import hppfcl
from pinocchio.visualize import MeshcatVisualizer

import numpy as np
import transformations as tf

In [58]:
# motoman = motoman_config.MotomanSDA10F()
robot_urdf = "../../robots/robotiq_arg85_description.URDF"
package_dirs = []
package_dirs.append("../../../robotiq_arg85_description")
package_dirs.append("../../../")

model, collision_model, visual_model = pin.buildModelsFromUrdf(robot_urdf, package_dirs=package_dirs)

# Start a new MeshCat server and client.
# Note: the server can also be started separately using the "meshcat-server" command in a terminal:
# this enables the server to remain active after the current script ends.
#
# Option open=True pens the visualizer.
# Note: the visualizer can also be opened seperately by visiting the provided URL.    
viz = MeshcatVisualizer(model, collision_model, visual_model)
viz.initViewer(open=True)

# Load the robot in the viewer.
viz.loadViewerModel()

# Display a robot configuration.
q0 = pin.neutral(model)
print('q0: ')
print(q0)
viz.display(q0)


You can open the visualizer by visiting the following URL:
http://127.0.0.1:7008/static/
q0: 
[0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0.]


In [59]:
pcd_total = []
# shelf-bottom
num_points = 5000
position = np.array([0.85, 0, 0.5])
half_size = np.array([0.175, 0.5, 0.5])
pcd = np.random.uniform(low=position-half_size, high=position+half_size, size=(num_points, 3))
pcd_total.append(pcd)
# shelf-top
num_points = 5000
position = np.array([0.85, 0, 1.42])
half_size = np.array([0.175, 0.5, 0.025])
pcd = np.random.uniform(low=position-half_size, high=position+half_size, size=(num_points, 3))
pcd_total.append(pcd)
# shelf-padding-left
num_points = 5000
position = np.array([0.85, -0.475, 1.2])
half_size = np.array([0.175, 0.025, 0.2])
pcd = np.random.uniform(low=position-half_size, high=position+half_size, size=(num_points, 3))
pcd_total.append(pcd)
# shelf-padding-right
num_points = 5000
position = np.array([0.85, 0.475, 1.2])
half_size = np.array([0.175, 0.025, 0.2])
pcd = np.random.uniform(low=position-half_size, high=position+half_size, size=(num_points, 3))
pcd_total.append(pcd)
# shelf-padding-back
num_points = 5000
position = np.array([1.0, 0, 1.2])
half_size = np.array([0.025, 0.5, 0.2])
pcd = np.random.uniform(low=position-half_size, high=position+half_size, size=(num_points, 3))
pcd_total.append(pcd)
pcd_total = np.concatenate(pcd_total, axis=0)


# add pcd to fcl
fcl_octree = hppfcl.makeOctree(pcd_total, 0.01)
# add fcl_pcd to pinocchio
octree_obj = pin.GeometryObject('octree', 0, fcl_octree, pin.SE3.Identity())
octree_obj.meshColor[0] = 1.0
collision_model.addGeometryObject(octree_obj)
# visual_model.addGeometryObject(octree_obj)

# add point cloud for visualization
point_cloud = hppfcl.BVHModelOBBRSS()
point_cloud.beginModel(0, len(pcd_total))
point_cloud.addVertices(pcd_total)
bvh_obj = pin.GeometryObject('bvh', 0, point_cloud, pin.SE3.Identity())
bvh_obj.meshColor[0] = 1.0
visual_model.addGeometryObject(bvh_obj)



9

In [ ]:
if False: pose = np.array([[-0.00718494,  0.28502675,  0.95849263,  0.96544325],
 [-0.01456196,  0.95838591, -0.28510418,  0.1875001 ],
 [-0.99986815, -0.01600599, -0.0027354,   1.0416948 ],
 [ 0.,          0.,          0.,          1.        ]]
)
pose = np.array([
    [0.00422704, 0.56854313, 0.82264259, 0.95622154],
    [0.01581709, -0.82258504, 0.56842208, 0.02153473],
    [0.99986597, 0.01060907, -0.0124698, 1.12221966],
    [0., 0., 0., 1.]
])

#EE_OFFSET = tf.identity_matrix()
#EE_OFFSET[:3, 3] = [0, 0, -.135]
EE_OFFSET = np.array([
    [1.00000000e+00, 4.42672396e-06, 5.22697465e-06, -1.29457946e-06],
    [-4.42674501e-06, 1.00000000e+00, 4.02734961e-06, 2.03632196e-07],
    [-5.22695682e-06, -4.02737275e-06, 1.00000000e+00, 7.61974474e-02],
    [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 1.00000000e+00]
])

pose @= EE_OFFSET

quat_wxyz = np.roll(tf.quaternion_from_matrix(pose), -1)
xyz = pose[:3, 3]

temp = pin.neutral(model)
if False:
    temp[3:7] = [0, 0.70710678, 0, 0.70710678]
    temp[2] = 1.2
    temp[1] = -.2
    temp[0] = .5
else: 
    temp[3:7] = quat_wxyz
    temp[:3] = xyz

viz.rebuildData()
viz.loadViewerModel()

viz.display(temp)

/home/kchen/Robotics/venv/lib/python3.11/site-packages/cmeel.prefix/lib/python3.11/site-packages/pinocchio/visualize/meshcat_visualizer.py:493: UserWarning: The geometry object named octree is not supported by Pinocchio/MeshCat for vizualization.
  self.loadViewerGeometryObject(collision, pin.GeometryType.COLLISION, color)


In [17]:
for a in collision_model.geometryObjects:
    print(a.__dir__())

['__module__', '__doc__', '__reduce__', '__init__', 'meshScale', 'meshColor', 'geometry', 'name', 'parentJoint', 'parentFrame', 'placement', 'meshPath', 'overrideMaterial', 'meshTexturePath', 'disableCollision', 'meshMaterial', '__eq__', '__ne__', 'CreateCapsule', '__new__', '__weakref__', '__dict__', '__repr__', '__hash__', '__str__', '__getattribute__', '__setattr__', '__delattr__', '__lt__', '__le__', '__gt__', '__ge__', '__reduce_ex__', '__getstate__', '__subclasshook__', '__init_subclass__', '__format__', '__sizeof__', '__dir__', '__class__']
['__module__', '__doc__', '__reduce__', '__init__', 'meshScale', 'meshColor', 'geometry', 'name', 'parentJoint', 'parentFrame', 'placement', 'meshPath', 'overrideMaterial', 'meshTexturePath', 'disableCollision', 'meshMaterial', '__eq__', '__ne__', 'CreateCapsule', '__new__', '__weakref__', '__dict__', '__repr__', '__hash__', '__str__', '__getattribute__', '__setattr__', '__delattr__', '__lt__', '__le__', '__gt__', '__ge__', '__reduce_ex__', '

In [45]:
for a in model.nqs:
    print(a)

0
7
1
1
1
1
1
1


In [46]:
model.lowerPositionLimit

array([ -inf,  -inf,  -inf, -1.01, -1.01, -1.01, -1.01,  0.  ,  0.  ,
        0.  ,  0.  ,  0.  ,  0.  ])

In [47]:
model.upperPositionLimit

array([   inf,    inf,    inf, 1.01  , 1.01  , 1.01  , 1.01  , 0.725 ,
       0.8757, 0.8757, 0.8757, 0.8757, 0.8757])

In [30]:
pin.neutral(model)

array([0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0.])

In [49]:
q1 = np.array([0, 0, 0, 0, 0, 0, 1] + [0]*6)#[0.72, 0.875, 0.875, 0.875, 0.875, 0.875]
print('q1: ')
print(q1)
viz.display(q1)

q1: 
[0 0 0 0 0 0 1 0 0 0 0 0 0]
